# ATLAS Solar IDW Interpolation Tutorial

This notebook interpolates station-based solar radiation observations onto the high-resolution grid produced by the downscaling workflow.

The notebook is organised as a tutorial:
1. set the input parameters;
2. load the downscaled grid coordinates;
3. load and prepare station observations;
4. apply Inverse Distance Weighting (IDW);
5. save the monthly gridded outputs.

All paths are relative and can be adapted by each country team.

## Methodology Description:
This notebook uses the **Inverse Distance Weighting (IDW)** interpolation method to transform point observations from meteorological stations into a continuous high-resolution raster surface.

The input dataset consists of **scattered station measurements**, where each station provides the observed value of solar radiation at a specific location. The objective is to estimate solar radiation at the unknown cells of the final **90 m resolution grid**.

For each grid cell, the interpolated value is computed as a weighted average of the surrounding stations. The weights are assigned according to the inverse of the distance between the grid cell and each station, giving greater importance to nearby observations. In the current implementation, the weight is proportional to the inverse cube of the distance:

$$
w_i = \frac{1}{d_i^3}
$$

where:

* $w_i$ is the weight assigned to station *i*
* $d_i$ is the distance between the interpolation point and station *i*

The interpolated value is then calculated as:

$$
\hat{x} = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}
$$

where:

* $\hat{x}$ is the predicted value at the target grid cell
* $x_i$ is the observed value at station *i*
* $w_i$ is the corresponding distance-based weight

As a result, stations located closer to the target location have a much stronger influence on the predicted value than more distant stations. This approach allows the generation of a continuous 90 m resolution raster while preserving the spatial patterns represented by the station observations and ensuring that local measurements have the greatest influence on nearby areas.

## Step 1 - Input parameters

Edit only this cell before running the notebook.

Expected folders:
- station observations: `../data/stations/{country}/`
- downscaled grid coordinates: output folder from the downscaling notebook
- IDW output: `../data/stations/{country}/idw/`

In [1]:
from pathlib import Path

# Country name used in folder and file names.
country = "argentina"

# Target variable used in the downscaling output folder and file names.
# For this workflow, the solar radiation variable is generally "ssrd".
target = "ssrd"

# Name of the variable column in the station CSV file.
# Change this only if your station file uses a different column name.
station_value_column = "ssrd"

# Unit of the station data.
# Supported options:
# - "W/m2": converted to kWh/m2/day
# - "kWh/m2/day": used as provided
unit = "W/m2"

# Months to process.
# Use a single month, e.g. [1], or all months as shown below.
months_to_process = list(range(1, 2))
month=1
# Input folder containing station observations.
input_path = Path(f"../data/stations/{country}/")

# CSV file containing station data.
station_file = input_path / f"allstats_solar_radiation_{country}.csv"

# Folder containing the downscaled files from the previous workflow.
coords_path = Path(f"../data/downscaled_data/{target}/{country}/")

# Template of the downscaled files used to read the target grid coordinates.
# The notebook reads latitude and longitude from one of these files.
coords_file_template = f"{target}_downscaled_{country}_m{month}.nc"

# Output folder for IDW products.
output_path = Path(f"../data/stations/{country}/idw/")
output_path.mkdir(parents=True, exist_ok=True)

## Step 2 - Import libraries

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr

from scipy.spatial import distance_matrix

## Step 3 - Helper functions

These functions read the downscaled grid, prepare station observations, apply IDW, and save the output NetCDF files.

In [3]:
def get_existing_coords_file(coords_folder, template, months):
    """Return the first available downscaled file used to read the target grid."""
    for month in months:
        candidate = coords_folder / template.format(month=month)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"No coordinate file found in {coords_folder}. "
        f"Expected files following this template: {template}"
    )


def read_target_grid(coords_file):
    """Read latitude and longitude from a downscaled NetCDF file."""
    ds = xr.open_dataset(coords_file)

    if "longitude" not in ds or "latitude" not in ds:
        raise KeyError("The coordinate file must contain 'longitude' and 'latitude' variables.")

    lon = ds["longitude"].values
    lat = ds["latitude"].values

    lon = lon[~np.isnan(lon)]
    lat = lat[~np.isnan(lat)]

    # Start from the northernmost row.
    lat = np.sort(np.unique(lat))[::-1]
    lon = np.sort(np.unique(lon))

    ds.close()
    return lat, lon


def read_station_dataset(filepath):
    """Read station observations from CSV and parse the time column when available."""
    if not Path(filepath).exists():
        raise FileNotFoundError(f"Station file not found: {filepath}")

    df = pd.read_csv(filepath)

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
    elif "month" not in df.columns:
        raise ValueError("The station file must contain either a 'time' column or a 'month' column.")

    required_columns = {"name", "latitude", "longitude", station_value_column}
    missing = required_columns.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in station file: {sorted(missing)}")

    return df


def prepare_station_monthly_data(group, value_column, input_unit):
    """Convert station observations to monthly mean kWh/m2/day values."""

    input_unit = input_unit.strip().lower()

    if "time" in group.columns:
        hourly = group.resample("60min", on="time").mean(numeric_only=True)
        daily = hourly.reset_index().resample("D", on="time").mean(numeric_only=True)
        daily = daily.reset_index()
    else:
        daily = group.copy()

    if input_unit in ["w/m2", "w m-2", "w/m²"]:
        daily[value_column] = (daily[value_column] * 24) / 1000

    elif input_unit in ["kwh/m2/day", "kwh m-2 day-1", "kwh/m²/day"]:
        pass

    else:
        raise ValueError(
            "Unsupported unit. Use 'W/m2' or 'kWh/m2/day'."
        )

    if "time" in daily.columns:
        monthly = daily.resample("M", on="time").mean(numeric_only=True)
        monthly = monthly.groupby(monthly.index.month).mean(numeric_only=True)

    elif "month" in daily.columns:
        columns_to_drop = [col for col in ["Unnamed: 0"] if col in daily.columns]
        monthly = daily.drop(columns=columns_to_drop).set_index("month")

    else:
        raise ValueError("The station data must contain either 'time' or 'month'.")

    monthly = monthly[monthly[value_column] > 0.5]

    return monthly


def simple_idw(pos, values, target_pos, power=3):
    """Apply Inverse Distance Weighting from station points to target grid points."""
    distances = distance_matrix(pos, target_pos)

    # Avoid division by zero when a target grid point matches a station location.
    distances = np.where(distances == 0, 1e-12, distances)

    weights = 1.0 / distances**power
    weights = weights / np.nansum(weights, axis=0)

    return np.dot(weights.T, values)


def apply_idw(station_month_df, lat_new, lon_new, value_column):
    """Interpolate station values to the target grid using IDW."""
    lat = station_month_df["latitude"].values
    lon = station_month_df["longitude"].values
    values = station_month_df[value_column].values

    station_positions = np.array(list(zip(lat, lon)))
    target_grid = np.array([[lat_value, lon_value] for lat_value in lat_new for lon_value in lon_new])

    interpolated = simple_idw(station_positions, values, target_grid)
    return interpolated.reshape(len(lat_new), len(lon_new))


def save_netcdf(ds, output_file):
    """Save a dataset as a compressed NetCDF file."""
    encoding = {var: {"zlib": True, "complevel": 1} for var in ds.data_vars}
    ds.to_netcdf(output_file, engine="netcdf4", encoding=encoding)


def run_idw_for_month(station_monthly_df, month, lat_grid, lon_grid, value_column, country_name, out_folder):
    """Run IDW for one month and save the result."""
    month_df = station_monthly_df[station_monthly_df.index == month]

    if month_df.empty:
        raise ValueError(f"No station data available for month {month}.")

    idw_array = apply_idw(month_df, lat_grid, lon_grid, value_column)

    ds = xr.DataArray(
        idw_array,
        coords=[("latitude", lat_grid), ("longitude", lon_grid)],
        name=f"{target}_reshaped",
        attrs={"units": "kWh/m2/day"},
    ).to_dataset()

    output_file = out_folder / f"{target}_idw_{country_name}_m{month}.nc"
    save_netcdf(ds, output_file)

    return output_file

## Step 4 - Load the target grid and station observations

In [4]:
coords_file = get_existing_coords_file(coords_path, coords_file_template, months_to_process)
lat_grid, lon_grid = read_target_grid(coords_file)

station_df = read_station_dataset(station_file)

station_monthly_df = station_df.groupby("name", group_keys=False).apply(
    lambda group: prepare_station_monthly_data(group, station_value_column, unit)
)
station_monthly_df

ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


,ssrd,name,latitude,longitude
month,,,,
1,11.196588,Bariloche,-41.148840,-71.162800
2,10.266176,Bariloche,-41.148840,-71.162800
3,8.983009,Bariloche,-41.148840,-71.162800
4,6.852621,Bariloche,-41.148840,-71.162800
5,5.201899,Bariloche,-41.148840,-71.162800
...,...,...,...,...
8,2.990529,Ushuaia,-54.847713,-68.307876
9,4.681875,Ushuaia,-54.847713,-68.307876
10,5.212279,Ushuaia,-54.847713,-68.307876


## Step 5 - Run IDW interpolation

This cell creates one NetCDF file per selected month in:

`../data/stations/{country}/idw/`

In [5]:
created_files = []

for month in months_to_process:
    output_file = run_idw_for_month(
        station_monthly_df=station_monthly_df,
        month=month,
        lat_grid=lat_grid,
        lon_grid=lon_grid,
        value_column=station_value_column,
        country_name=country,
        out_folder=output_path,
    )
    created_files.append(output_file)

created_files

[PosixPath('../data/stations/argentina/idw/ssrd_idw_argentina_m1.nc')]

## Notes for country teams

Before running the notebook, check that:
- the station CSV contains `name`, `latitude`, `longitude`, and the station variable column;
- the station CSV contains either `time` or `month`;
- the downscaling output folder contains at least one monthly file with latitude and longitude variables;
- the unit is correctly set before the conversion step.